Alignment in Napari

In [ ]:
%pip install napari_spatialdata[all]
%pip install spatialdata
%pip install sopa

In [ ]:
%pip install --upgrade napari-spatialdata spatialdata napari

In [ ]:
#in case of QtBindingsNotFoundError #probably not needed if using %install napari_spatialdata[all]
%pip install PyQt5

In [1]:
import napari
import spatialdata 
import sopa
import napari_spatialdata
import spatialdata_plot 

d:\Dom\Virtual_Environments\napari_registration_project\.venv\Lib\site-packages\dask\dataframe\__init__.py:31: FutureWarning: The legacy Dask DataFrame implementation is deprecated and will be removed in a future version. Set the configuration option `dataframe.query-planning` to `True` or None to enable the new Dask Dataframe implementation and silence this warning.
  warnings.warn(
d:\Dom\Virtual_Environments\napari_registration_project\.venv\Lib\site-packages\xarray_schema\__init__.py:1: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import DistributionNotFound, get_distribution
d:\Dom\Virtual_Environments\napari_registration_project\.venv\Lib\site-packages\spatialdata\_core\query\relational_query.py:530: FutureWarning: functools.partial will be a method descriptor in fu

In [ ]:
xen_image = spatialdata.read_zarr("")

In [ ]:
#Create spatial objects containing the images you want to align

#eventually need these to contain every xen/pheno channel. Will start by registering xenDapi and one pheno channel to get the affine, as trying all of them may cause memory issues. 


xen_image = sopa.io.ome_tif("D:/Dom/Fibrosis project/4th year data - update location/pheno_xen_fullres_images/morphology_focus_scaledX_dapi.ome.tif", as_image = False) #ensure 'as.image = False', otherwise you create an image not a spatialdata object --> could do true, then create a dictionary with all images, then add them to a single spatial data object in future. 
pheno_image = sopa.io.ome_tif("D:/Dom/Fibrosis project/4th year data - update location/pheno_xen_fullres_images/pheno_C12.ome.tif", as_image = False) #12th channel

#save to a zarr file for easy saving/organisation/reading etc.
xen_image.write("D:/Dom/Fibrosis Project/4th year data - update location/xen_pheno_registration/xen_dapi_sd.zarr", overwrite = True)
pheno_image.write("D:/Dom/Fibrosis Project/4th year data - update location/xen_pheno_registration/pheno_sd.zarr", overwrite = True)


print(xen_image)
print(pheno_image)


In [ ]:
#reading in pre-made spatial objects
xen_image = spatialdata.read_zarr("D:/Dom/Fibrosis Project/4th year data - update location/xen_pheno_registration/xen_dapi_sd.zarr")
pheno_image = spatialdata.read_zarr("D:/Dom/Fibrosis Project/4th year data - update location/xen_pheno_registration/pheno_sd.zarr")

xen_masks_cells = spatialdata.read_zarr("D:/Dom/Fibrosis Project/4th year data - update location/xen_pheno_registration/xen_masks_cells_sd.zarr")
xen_masks_both = spatialdata.read_zarr("D:/Dom/Fibrosis Project/4th year data - update location/xen_pheno_registration/xen_masks_both_sd.zarr")


print(xen_image)
print(pheno_image)


In [ ]:
#go to 'global' in bottom left and click, then open images from elements section. 
from napari_spatialdata import Interactive

interactive = Interactive([xen_image, pheno_image])
interactive.run()

#make points layers, name, then select landmarks
#save points layers as csvs files for next section


In [ ]:
#add points_layers.csv files to spatialdata objects

import pandas as pd
import numpy as np
from spatialdata.models import PointsModel
from spatialdata.transformations import Identity

# Read your CSV file
landmarks_df = pd.read_csv('D:/Dom/Fibrosis Project/4th year data - update location/xen_pheno_registration/xen_landmarks.csv')
pheno_landmarks_df = pd.read_csv('D:/Dom/Fibrosis Project/4th year data - update location/xen_pheno_registration/pheno_landmarks.csv')

# Extract coordinates (axis-1 = x, axis-2 = y)
# SpatialData expects coordinates in (y, x) order for 2D points
coords = landmarks_df[['axis-2', 'axis-1']].values
pheno_coords = pheno_landmarks_df[['axis-2', 'axis-1']].values

# Create a DataFrame in the format expected by PointsModel
# PointsModel expects columns: ['x', 'y'] or ['z', 'y', 'x'] for 3D
points_df = pd.DataFrame({
    'y': landmarks_df['axis-1'].values,
    'x': landmarks_df['axis-2'].values
})

pheno_points_df = pd.DataFrame({
    'y': pheno_landmarks_df['axis-1'].values,
    'x': pheno_landmarks_df['axis-2'].values
})

# Create the points element with transformation to global coordinate system
points_element = PointsModel.parse(
    points_df,
    transformations={"global": Identity()}
)

pheno_points_element = PointsModel.parse(
    pheno_points_df,
    transformations={"global": Identity()}
)

# Add to your xen_image SpatialData object
xen_image.points['xen_landmarks'] = points_element
pheno_image.points['pheno_landmarks'] = pheno_points_element 


print(xen_image)
print(pheno_image)

In [ ]:
from spatialdata.transformations import (
    align_elements_using_landmarks,
    get_transformation_between_landmarks,
)

affine = get_transformation_between_landmarks(
    references_coords=xen_image["xen_landmarks"], moving_coords=pheno_image["pheno_landmarks"]
)
affine
affine = align_elements_using_landmarks(
    references_coords=xen_image["xen_landmarks"], 
    moving_coords=pheno_image["pheno_landmarks"],
    reference_element=xen_image["morphology_focus_scaledX_dapi"],
    moving_element=pheno_image["pheno_C12"],
    reference_coordinate_system="global",
    moving_coordinate_system="global",
    new_coordinate_system="aligned",
)
affine


In [ ]:
#Saving the landmarks and alignment
xen_image.write_element("xen_landmarks")
pheno_image.write_element("pheno_landmarks")

xen_image.write_transformations()
pheno_image.write_transformations()

In [ ]:
#adding a new image to the aligned coordinate system. 

pheno_c1 = sopa.io.ome_tif("D:/Dom/Fibrosis project/4th year data - update location/pheno_xen_fullres_images/pheno_C1.ome.tif")

image = pheno_c1.images['pheno_C1']

pheno_image['pheno_C1'] = image

aligned_pheno_c1= pheno_image.transform_element_to_coordinate_system('pheno_C1', 'aligned', maintain_positioning = False)
pheno_image['pheno_C1'] = aligned_pheno_c1
print(pheno_image)

#saving the result
pheno_image.write_element("pheno_C1")
print(pheno_image)

In [ ]:
image = xen_masks_cells.images['cells_segmentation_mask0']

xen_image['cells_mask'] = image

aligned_xen_masks = xen_image.transform_element_to_coordinate_system('cells_mask', 'aligned', maintain_positioning = False)
xen_image['cells_mask'] = aligned_xen_masks
xen_image.write_element('cells_mask')
print(xen_image)